In [1]:
import pandas as pd

# Load data training
df = pd.read_csv('train.csv')

# 1. Cek 5 baris data pertama
print(" 5 Baris pertama")
display(df.head())

# 2. Cek indo kolom, tipe data dan missing values
print("\n - INFO DATASET - ")
df.info

 5 Baris pertama


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S



 - INFO DATASET - 


<bound method DataFrame.info of      PassengerId  Survived  Pclass  \
0              1         0       3   
1              2         1       1   
2              3         1       3   
3              4         1       1   
4              5         0       3   
..           ...       ...     ...   
886          887         0       2   
887          888         1       1   
888          889         0       3   
889          890         1       1   
890          891         0       3   

                                                  Name     Sex   Age  SibSp  \
0                              Braund, Mr. Owen Harris    male  22.0      1   
1    Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                               Heikkinen, Miss. Laina  female  26.0      0   
3         Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                             Allen, Mr. William Henry    male  35.0      0   
..                                   

In [2]:
#Cek jumlah missing values
missing_data = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

# Gabungin jadi satu tabel ringkas
df_missing = pd.DataFrame({
    'Jumlah Missing Values': missing_data,
    'Persentase (%)': missing_percent
})

# Tampilkan cuma kolom yang ada missing values
print(" - Kolom dengan Missing Values - ")
display(df_missing[df_missing['Jumlah Missing Values'] > 0])

 - Kolom dengan Missing Values - 


,Jumlah Missing Values,Persentase (%)
Age,177,19.865320
Cabin,687,77.104377
Embarked,2,0.224467


In [3]:
# 1, Hapus kolom yang tidak diperlukan atau terlalu banyak hilang
# Cabin hilang 77%, PassengeId/ Name / Ticket tidak terlalu berpengaruh terhadap prediksi, jadi dihapus
df_clean = df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# 2. Isi missing values pada kolom Age dengan median
median_age = df_clean['Age'].median()
df_clean['Age'] = df_clean['Age'].fillna(median_age)

# 3. Isi missing values pada kolom Embarked dengan modus
mode_embarked = df_clean['Embarked'].mode()[0]
df_clean['Embarked'] = df_clean['Embarked'].fillna(mode_embarked)

# 4. Verifikasi ulang missing values
print(" - SISA MISSING VALUES - ")
print(df_clean.isnull().sum())

# 5. Cek 5 baris data pertama setelah pembersihan
display(df_clean.head())


 - SISA MISSING VALUES - 
Survived    0
Pclass      0
Sex         0
Age         0
SibSp       0
Parch       0
Fare        0
Embarked    0
dtype: int64


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [4]:
# 1. Encoding Sex ( Male =  1, Female = 0)
df_clean['Sex'] = df_clean['Sex'].map({'male': 1, 'female': 0})

# 2. One-hot encoding untuk kolom Embarked
df_clean =  pd.get_dummies(df_clean, columns=['Embarked'], drop_first=True, dtype=int)

# Cek Hasil transformasi data
print(" - Data setelah encoding - ")
display(df_clean.head())

 - Data setelah encoding - 


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_Q,Embarked_S
0,0,3,1,22.0,1,0,7.2500,0,1
1,1,1,0,38.0,1,0,71.2833,0,0
2,1,3,0,26.0,0,0,7.9250,0,1
3,1,1,0,35.0,1,0,53.1000,0,1
4,0,3,1,35.0,0,0,8.0500,0,1


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1, Pisahkan fitur x dan target y
X = df_clean.drop(columns=['Survived'])
y = df_clean['Survived']

# 2. Split data : 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split (X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Scalling fitur angka (agar fitur skala besar seperti 'Fare' tidak mendominasi 'Age')
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Jumlah data training: {X_train.shape[0]} baris")
print(f"Jumlah data testing: {X_test.shape[0]} baris")

Jumlah data training: 712 baris
Jumlah data testing: 179 baris


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Inisialisasi dan Train model
model = RandomForestClassifier(random_state=42)
model.fit(X_train_scaled, y_train)

# 2. Prediksi data testing
y_pred = model.predict(X_test_scaled)

# 3. Evaluasi model
accuracy = accuracy_score(y_test, y_pred)
print(f" Akurasi model : {accuracy * 100:.2f}% \n")

print(" - Classification Report - ")
print(classification_report(y_test, y_pred))

 Akurasi model : 80.45% 

 - Classification Report - 
              precision    recall  f1-score   support

           0       0.82      0.87      0.85       110
           1       0.77      0.70      0.73        69

    accuracy                           0.80       179
   macro avg       0.80      0.78      0.79       179
weighted avg       0.80      0.80      0.80       179

